In [1]:
%pip install folium pandas requests

  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached xyzservices-2026.3.0-py3-none-any.whl.metadata (4.1 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached xyzservices-2026.3.0-py3-none-any.whl (94 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import folium
from folium import plugins
import json
import requests
import time
import os
import pandas as pd
import re

# 1. Warna Rute Diubah ke Versi Gelap / Bold
warna_wilayah = {
    "pusat": "#c0392b",   # Merah Gelap
    "utara": "#2980b9",   # Biru Gelap
    "selatan": "#27ae60", # Hijau Gelap
    "timur": "#8e44ad",   # Ungu Gelap
    "barat": "#d4ac0d"    # kuning
}

# Tambahkan fungsi ini untuk membaca file CSV
def load_koordinat(path_csv):
    try:
        # Membaca data CSV menggunakan Pandas
        df = pd.read_csv(path_csv)
        koordinat_dict = {}
        
        # Looping setiap baris di CSV
        for index, row in df.iterrows():
            nama = str(row['Nama']).strip() 
            koordinat_dict[nama] = [row['Latitude'], row['Longitude']]
            
        return koordinat_dict
    except FileNotFoundError:
        print(f"Error: File {path_csv} tidak ditemukan! Pastikan path/lokasi file benar.")
        return {}
    except KeyError as e:
        print(f"Error: Kolom {e} tidak ditemukan di CSV. Cek nama header CSV kamu.")
        return {}

# Tambahkan fungsi ini untuk menarik garis rute jalan dari API OSRM
def get_real_route_osrm(coord_asal, coord_tujuan):
    # OSRM menggunakan format (Longitude, Latitude), sedangkan Folium (Latitude, Longitude)
    lon_asal, lat_asal = coord_asal[1], coord_asal[0]
    lon_tujuan, lat_tujuan = coord_tujuan[1], coord_tujuan[0]
    
    url = f"http://router.project-osrm.org/route/v1/driving/{lon_asal},{lat_asal};{lon_tujuan},{lat_tujuan}?overview=full&geometries=geojson"
    
    try:
        r = requests.get(url)
        r.raise_for_status() # Memastikan tidak ada error dari server
        res = r.json()
        
        if res['code'] == 'Ok':
            # Mengambil data rute
            routes = res['routes'][0]
            jarak_m = routes['distance'] # Jarak dalam meter
            geometry = routes['geometry']['coordinates']
            
            # Balikkan lagi format koordinat menjadi (Latitude, Longitude) untuk Folium
            rute_nyata = [[point[1], point[0]] for point in geometry]
            return rute_nyata, jarak_m
            
    except Exception as e:
        print(f"Gagal mengambil rute OSRM: {e}")
        
    # Fallback: Jika server OSRM gagal, kembalikan garis lurus biasa dan jarak 0
    return [coord_asal, coord_tujuan], 0

def buat_peta_rute():
    path_csv = '../data/koordinat_puskesmas.csv'
    koordinat_lokasi = load_koordinat(path_csv)
    
    if len(koordinat_lokasi) == 0:
        return None

    nama_gudang = "UPTD Gudang Farmasi Surabaya"
    titik_pusat = koordinat_lokasi.get(nama_gudang, list(koordinat_lokasi.values())[0])

    # Menggunakan tema peta terang (Light Mode)
    peta = folium.Map(location=titik_pusat, zoom_start=12, tiles="CartoDB positron", zoom_control=False)
    plugins.LocateControl().add_to(peta)

    # Ikon start diubah ke pin biasa (map-marker)
    if nama_gudang in koordinat_lokasi:
        folium.Marker(
            location=titik_pusat,
            popup="<b>UPTD Gudang Farmasi Surabaya</b><br>Titik Awal & Akhir",
            icon=folium.Icon(color='red', icon='map-marker', prefix='fa')
        ).add_to(peta)

    algoritma_files = {
        "ga": {"nama": "Algoritma GA", "file_json": "../output_json/rute_ga.json"},
        "ma": {"nama": "Algoritma MA", "file_json": "../output_json/rute_ma.json"},
        "aco": {"nama": "Algoritma ACO", "file_json": "../output_json/rute_aco.json"},
        "pso": {"nama": "Algoritma PSO", "file_json": "../output_json/rute_dpso.json"}
    }

    jarak_optimasi_otomatis = {"ga": {}, "ma": {}, "aco": {}, "pso": {}}
    rute_per_algo_wilayah = {"ga": {}, "ma": {}, "aco": {}, "pso": {}}
    
    print("Membaca data rute dan jarak dari file JSON...")

    for id_algo, info in algoritma_files.items():
        filepath = info["file_json"]
        nama_algo = info["nama"]
        
        if not os.path.exists(filepath):
            continue
            
        with open(filepath, 'r') as f:
            data_rute = json.load(f)
            
        for wilayah, data_wilayah in data_rute.items():
            wilayah_lower_key = "lainnya"
            warna_rute = "#bdc3c7"
            
            for key, warna in warna_wilayah.items():
                if key in wilayah.lower():
                    wilayah_lower_key = key
                    warna_rute = warna
                    break
            
            class_identitas = f"algo-{id_algo} wil-{wilayah_lower_key}"
            
            if isinstance(data_wilayah, dict) and "rute_nama" in data_wilayah:
                urutan_lokasi = data_wilayah["rute_nama"]
                if "jarak_optimasi" in data_wilayah:
                    jarak_optimasi_otomatis[id_algo][wilayah_lower_key] = data_wilayah["jarak_optimasi"]
            else:
                urutan_lokasi = data_wilayah

            rute_per_algo_wilayah[id_algo][wilayah_lower_key] = urutan_lokasi

            for i in range(len(urutan_lokasi) - 1):
                lokasi_asal = urutan_lokasi[i].strip()
                lokasi_tujuan = urutan_lokasi[i+1].strip()
                
                if lokasi_asal in koordinat_lokasi and lokasi_tujuan in koordinat_lokasi:
                    coord_asal = koordinat_lokasi[lokasi_asal]
                    coord_tujuan = koordinat_lokasi[lokasi_tujuan]
                    
                    rute_nyata, jarak_m = get_real_route_osrm(coord_asal, coord_tujuan)
                    
                    # Garis Rute Gelap
                    folium.PolyLine(
                        locations=rute_nyata,
                        color=warna_rute,
                        weight=5, 
                        opacity=0.9,
                        className=class_identitas,
                        tooltip=f"<span style='color:{warna_rute}; font-weight:bold;'>{nama_algo} - {wilayah}</span><br>{lokasi_asal} ➔ {lokasi_tujuan}"
                    ).add_to(peta)
                    
                    # Pembuatan marker angka urutan
                    if lokasi_tujuan != nama_gudang:
                        urutan_nomor = i + 1
                        
                        # Style HTML untuk bulatan angka
                        icon_html = f"""
                        <div style="
                            background-color: {warna_rute};
                            color: white;
                            width: 20px;
                            height: 20px;
                            border-radius: 50%;
                            text-align: center;
                            line-height: 18px;
                            font-size: 10px;
                            font-weight: bold;
                            border: 1.5px solid white;
                            box-shadow: 0 1px 3px rgba(0,0,0,0.4);
                        ">{urutan_nomor}</div>
                        """
                        
                        folium.Marker(
                            location=coord_tujuan,
                            popup=f"<b>{urutan_nomor}. {lokasi_tujuan}</b><br>Klaster: {wilayah}",
                            icon=folium.DivIcon(html=icon_html, class_name=class_identitas)
                        ).add_to(peta)
                    
                    time.sleep(0.02) 

    jarak_js = json.dumps(jarak_optimasi_otomatis)
    rute_js = json.dumps(rute_per_algo_wilayah)
    warna_js = json.dumps(warna_wilayah)

    # CSS dinamis khusus untuk pewarnaan hover/active tombol filter
    css_dynamic_buttons = ""
    for wil, warna in warna_wilayah.items():
        css_dynamic_buttons += f".btn-wil[data-wil='{wil}']:hover, .btn-wil.active-{wil} {{ background: {warna} !important; color: #ffffff !important; box-shadow: 0 3px 8px {warna}60; }}\n"

    # CSS & JS Dashboard - Tema Light + Layout Dirapatkan
    custom_dashboard = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&display=swap');
        
        /* Tema Light Dashboard */
        .custom-dashboard {{
            position: absolute; top: 15px; left: 15px; z-index: 9999;
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(8px);
            padding: 15px 18px;
            border-radius: 14px;
            box-shadow: 0px 8px 25px rgba(0, 0, 0, 0.15);
            font-family: 'Poppins', sans-serif; color: #2d3436; 
            width: 310px; border: 1px solid rgba(0,0,0,0.08);
            max-height: 92vh; display: flex; flex-direction: column;
        }}
        
        .dash-header {{ flex-shrink: 0; }}
        
        .dash-title {{ 
            font-size: 15px; font-weight: 700; margin: 0 0 10px 0; 
            letter-spacing: 0.5px; text-transform: uppercase; color: #2d3436; 
            border-bottom: 2px solid #f1f2f6; padding-bottom: 8px;
        }}
        
        .section-label {{ 
            font-size: 10px; font-weight: 600; color: #636e72; 
            margin-bottom: 5px; text-transform: uppercase; letter-spacing: 1px;
        }}
        
        .btn-container {{ display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 10px; }}
        
        .btn-pill {{
            padding: 5px 12px; border-radius: 20px; border: none; font-size: 11px; font-weight: 600;
            cursor: pointer; transition: all 0.2s ease; background: #f1f2f6; color: #636e72; 
            font-family: 'Poppins', sans-serif;
        }}
        
        /* Tombol Algoritma */
        .btn-algo:hover {{ background: #dfe6e9; color: #2d3436; }}
        .btn-algo.active {{ background: #0984e3; color: #ffffff; box-shadow: 0 3px 8px rgba(9, 132, 227, 0.3); }}
        
        /* Tombol Pilih Semua */
        #btn-semua-wil {{ margin-bottom: 10px; width: 100%; transition: all 0.2s ease; padding: 6px; }}
        
        /* Render Hover/Active Button Wilayah secara dinamis */
        {css_dynamic_buttons}

        /* Box Jarak - Lebih Kecil / Rapat */
        .distance-box {{ 
            background: #f8f9fa; border-radius: 10px; padding: 8px; 
            margin-top: 2px; text-align: center; border: 1px solid #e9ecef; 
            flex-shrink: 0;
        }}
        .dist-label {{ font-size: 11px; color: #636e72; font-weight: 600; display: block; }}
        .dist-value {{ font-size: 22px; font-weight: 700; color: #2d3436; line-height: 1.1; display: inline-block; margin-top: 3px; }}
        .dist-unit {{ font-size: 13px; font-weight: 600; color: #b2bec3; }}

        /* Bagian List Rute - Space diminimalkan */
        .route-list-container {{
            margin-top: 8px; overflow-y: auto; flex-grow: 1; 
            padding-right: 4px; display: flex; flex-direction: column; gap: 8px;
        }}
        
        /* Styling Scrollbar */
        .route-list-container::-webkit-scrollbar {{ width: 5px; }}
        .route-list-container::-webkit-scrollbar-track {{ background: #f1f2f6; border-radius: 10px; }}
        .route-list-container::-webkit-scrollbar-thumb {{ background: #b2bec3; border-radius: 10px; }}
        
        .route-group {{
            background: #ffffff; border-radius: 8px; padding: 8px 10px;
            border-left: 4px solid; border-top: 1px solid #f1f2f6; 
            border-right: 1px solid #f1f2f6; border-bottom: 1px solid #f1f2f6;
            box-shadow: 0 1px 3px rgba(0,0,0,0.02);
        }}
        
        .route-group-title {{
            font-size: 11px; font-weight: 700; text-transform: uppercase;
            margin-bottom: 6px; color: #2d3436;
        }}
        
        .route-steps {{
            list-style: none; padding: 0; margin: 0; position: relative;
        }}
        
        .route-steps::before {{
            content: ''; position: absolute; top: 10px; bottom: 10px; left: 6px;
            width: 2px; background: #dfe6e9; z-index: 1;
        }}
        
        .route-step-item {{
            font-size: 11px; padding: 3px 0 3px 20px; position: relative;
            color: #636e72; line-height: 1.3;
        }}
        
        /* Node titik di List (Warna background putih untuk efek Light) */
        .route-step-item::before {{
            content: ''; position: absolute; left: 2px; top: 6px;
            width: 10px; height: 10px; border-radius: 50%;
            background: #ffffff; border: 2.5px solid; z-index: 2;
        }}

        /* Sembunyikan elemen Leaflet berdasarkan filter (menggunakan class pada div/icon/polyline) */
        .hide-ga .algo-ga {{ display: none !important; opacity: 0 !important; }}
        .hide-ma .algo-ma {{ display: none !important; opacity: 0 !important; }}
        .hide-aco .algo-aco {{ display: none !important; opacity: 0 !important; }}
        .hide-pso .algo-pso {{ display: none !important; opacity: 0 !important; }}
        
        .hide-pusat .wil-pusat {{ display: none !important; opacity: 0 !important; }}
        .hide-utara .wil-utara {{ display: none !important; opacity: 0 !important; }}
        .hide-selatan .wil-selatan {{ display: none !important; opacity: 0 !important; }}
        .hide-timur .wil-timur {{ display: none !important; opacity: 0 !important; }}
        .hide-barat .wil-barat {{ display: none !important; opacity: 0 !important; }}
    </style>

    <div class="custom-dashboard" id="mainDashboard">
        <div class="dash-header">
            <h3 class="dash-title">📊 Rute Distribusi Farmasi</h3>
            
            <div class="section-label">Pilih Algoritma</div>
            <div class="btn-container">
                <button class="btn-pill btn-algo active" data-algo="ga">GA</button>
                <button class="btn-pill btn-algo" data-algo="ma">MA</button>
                <button class="btn-pill btn-algo" data-algo="aco">ACO</button>
                <button class="btn-pill btn-algo" data-algo="pso">PSO</button>
            </div>
            
            <div class="section-label">Filter Wilayah (Klaster)</div>
            <button class="btn-pill" id="btn-semua-wil"></button>
            <div class="btn-container">
                <button class="btn-pill btn-wil active-pusat" data-wil="pusat">Pusat</button>
                <button class="btn-pill btn-wil active-utara" data-wil="utara">Utara</button>
                <button class="btn-pill btn-wil active-selatan" data-wil="selatan">Selatan</button>
                <button class="btn-pill btn-wil active-timur" data-wil="timur">Timur</button>
                <button class="btn-pill btn-wil active-barat" data-wil="barat">Barat</button>
            </div>
            
            <div class="distance-box">
                <span class="dist-label">ESTIMASI JARAK OPTIMASI</span>
                <div><span class="dist-value" id="distDisplay">0.00</span> <span class="dist-unit">km</span></div>
            </div>
            
            <div class="section-label" style="margin-top: 10px;">Urutan Rute Kunjungan</div>
        </div>
        
        <div class="route-list-container" id="routeListContainer">
            <!-- Isi di-generate via JavaScript -->
        </div>
    </div>

    <script>
    setTimeout(function() {{
        const mapContainer = document.querySelector('.leaflet-container');
        const dataJarak = {jarak_js}; 
        const dataRute = {rute_js};
        const warnaWilayah = {warna_js};
        
        const algoBtns = document.querySelectorAll('.btn-algo');
        const wilBtns = document.querySelectorAll('.btn-wil');
        const btnSemuaWil = document.getElementById('btn-semua-wil');
        const distDisplay = document.getElementById('distDisplay');
        const routeListContainer = document.getElementById('routeListContainer');

        let activeAlgo = "ga";
        let activeWils = new Set(["pusat", "utara", "selatan", "timur", "barat"]);

        function updateSelectAllBtn() {{
            if (activeWils.size > 0) {{
                btnSemuaWil.innerText = "Hapus Semua Pilihan";
                btnSemuaWil.style.background = "#ff7675"; 
                btnSemuaWil.style.color = "#ffffff";
            }} else {{
                btnSemuaWil.innerText = "Pilih Semua Wilayah";
                btnSemuaWil.style.background = "#74b9ff"; 
                btnSemuaWil.style.color = "#ffffff";
            }}
        }}

        function generateRouteListUI() {{
            routeListContainer.innerHTML = ""; 
            
            if (!dataRute[activeAlgo]) return;

            const urutanWilayah = ["pusat", "utara", "selatan", "timur", "barat"];
            
            urutanWilayah.forEach(wil => {{
                if (activeWils.has(wil) && dataRute[activeAlgo][wil]) {{
                    const urutanTitik = dataRute[activeAlgo][wil];
                    const warna = warnaWilayah[wil] || "#bdc3c7";
                    
                    const groupDiv = document.createElement('div');
                    groupDiv.className = 'route-group';
                    groupDiv.style.borderLeftColor = warna;
                    
                    const titleDiv = document.createElement('div');
                    titleDiv.className = 'route-group-title';
                    titleDiv.innerText = "Klaster " + wil.charAt(0).toUpperCase() + wil.slice(1);
                    groupDiv.appendChild(titleDiv);
                    
                    const ulElement = document.createElement('ul');
                    ulElement.className = 'route-steps';
                    ulElement.style.setProperty('--step-color', warna);
                    
                    urutanTitik.forEach((titik, index) => {{
                        const liElement = document.createElement('li');
                        liElement.className = 'route-step-item';
                        
                        // Cek jika ini titik awal (0) atau akhir, hilangkan nomornya dari list ui
                        const isGudang = index === 0 || index === urutanTitik.length - 1;
                        let textPuskes = titik.replace("Surabaya", "").trim();
                        
                        // Menambahkan angka pada list tulisan agar sejajar dengan peta
                        liElement.innerText = isGudang ? textPuskes : `${{index}}. ${{textPuskes}}`; 
                        
                        const styleNode = document.createElement('style');
                        liElement.appendChild(styleNode);
                        liElement.style.cssText = `border-color: ${{warna}};`;
                        
                        ulElement.appendChild(liElement);
                    }});
                    
                    const styleBlock = document.createElement('style');
                    styleBlock.innerHTML = `.route-steps[style*="--step-color: ${{warna}}"] .route-step-item::before {{ border-color: ${{warna}}; }}`;
                    groupDiv.appendChild(styleBlock);
                    
                    groupDiv.appendChild(ulElement);
                    routeListContainer.appendChild(groupDiv);
                }}
            }});
        }}

        function updateDashboard() {{
            ['ga', 'ma', 'aco', 'pso'].forEach(a => {{
                if(a === activeAlgo) mapContainer.classList.remove('hide-' + a);
                else mapContainer.classList.add('hide-' + a);
            }});
            ['pusat', 'utara', 'selatan', 'timur', 'barat'].forEach(w => {{
                if(activeWils.has(w)) mapContainer.classList.remove('hide-' + w);
                else mapContainer.classList.add('hide-' + w);
            }});

            let totalKm = 0;
            if(dataJarak[activeAlgo]) {{
                activeWils.forEach(w => {{
                    if(dataJarak[activeAlgo][w]) {{
                        totalKm += dataJarak[activeAlgo][w];
                    }}
                }});
            }}
            distDisplay.innerText = totalKm.toFixed(2);
            generateRouteListUI();
        }}

        algoBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                algoBtns.forEach(b => b.classList.remove('active'));
                this.classList.add('active');
                activeAlgo = this.getAttribute('data-algo');
                updateDashboard();
            }});
        }});

        wilBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                const w = this.getAttribute('data-wil');
                const activeClass = 'active-' + w;
                if(activeWils.has(w)) {{
                    activeWils.delete(w);
                    this.classList.remove(activeClass);
                }} else {{
                    activeWils.add(w);
                    this.classList.add(activeClass);
                }}
                updateSelectAllBtn();
                updateDashboard();
            }});
        }});

        btnSemuaWil.addEventListener('click', function() {{
            if (activeWils.size > 0) {{
                activeWils.clear();
                wilBtns.forEach(btn => {{
                    btn.classList.remove('active-' + btn.getAttribute('data-wil'));
                }});
            }} else {{
                wilBtns.forEach(btn => {{
                    const w = btn.getAttribute('data-wil');
                    activeWils.add(w);
                    btn.classList.add('active-' + w);
                }});
            }}
            updateSelectAllBtn();
            updateDashboard();
        }});

        mapContainer.classList.add('hide-ma', 'hide-aco', 'hide-pso');
        updateSelectAllBtn();
        updateDashboard();

    }}, 500);
    </script>
    """
    peta.get_root().html.add_child(folium.Element(custom_dashboard))

    os.makedirs("../collab", exist_ok=True)
    output_html = "../collab/visualisasi_perbandingan_rute.html"
    peta.save(output_html)
    return peta

peta_hasil = buat_peta_rute()
peta_hasil

Membaca data rute dan jarak dari file JSON...
